# Task 5 — Capstone: Redesign One Chart for the Human Visual System

**Data:** `gapminder.csv` — 142 countries × 12 years (1952–2007), 5-year intervals.

**Question the chart should answer:** *Which continent had the fastest rise in life expectancy from 1952 to 2007?*

**Participant (human test):** a classmate (Sara), age 21. Shown the redesign only; had not seen the data before.

**Preattentive attribute used:** *hue* — Asia is highlighted in a single strong colour; all other continents are grey.  
**Gestalt principle used:** *continuity* — each line is a smooth path; the eye follows each continent's trajectory through the time axis.  
**Most important variable on highest-ranking channel:** `lifeExp` on *position* (y-axis) — the highest-ranking quantitative channel.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

gm = pd.read_csv('data/gapminder.csv')
print('Shape:', gm.shape)
print('Continents:', sorted(gm['continent'].unique()))
print('Years:', sorted(gm['year'].unique()))
gm.head()

In [ ]:
# ── Step 1: BAD EXAMPLE — spaghetti line chart, 142 countries ─────────────
fig, ax = plt.subplots(figsize=(14, 7))

for country in gm['country'].unique():
    sub = gm[gm['country'] == country].sort_values('year')
    ax.plot(sub['year'], sub['lifeExp'],
            linewidth=0.6, alpha=0.5)

ax.set_xlabel('year')
ax.set_ylabel('lifeExp')
ax.set_title('BAD EXAMPLE: Life expectancy over time — year vs lifeExp (n=142 countries)',
             loc='left', fontsize=11)

plt.tight_layout()
plt.savefig('bad_spaghetti.png', dpi=300, bbox_inches='tight')
plt.show()

print('Question this chart should answer: Which continent rose fastest in life expectancy?')
print('Actual result: impossible to answer — 142 lines with no grouping, 14+ colours,')
print('no pop-out element, and no message in the title.')

## Perception Audit — BAD EXAMPLE (before redesign)

| Row | Analysis |
|---|---|
| **Preattentive** | Every line is a different auto-assigned colour. No single attribute pops. Every line competes equally. *None* of the colour is carrying useful information about the continent grouping. |
| **Pop-out** | Nothing pops out. There is no single element the eye is drawn to. The title claims nothing specific, so there is no intended pop-out. |
| **Gestalt** | *Continuity* is present (lines read as time series) but *similarity* fires on 142 arbitrary colours, creating 142 apparent groups instead of 5 continent groups. The principle triggered is wrong — it groups by country instead of by continent. |
| **Cognitive load** | *Intrinsic:* the trend data itself (necessary). *Extraneous:* 142 colours; no legend possible; title names axes not finding; 142 paths the reader must trace individually. Reader must hold all 142 countries in working memory to synthesise an answer — vastly over the 4 ± 1 limit. |
| **Channels** | `lifeExp` → position (y), which is correct. `year` → position (x), also correct. `country` → colour, which fails: 142 colours cannot be distinguished and no legend fits. |
| **Working memory** | Reader must hold 142 country-colour pairs to read the legend (which does not fit). Chunking: none. Far over the 4 ± 1 limit. |

In [ ]:
# ── Step 3: Redesign ──────────────────────────────────────────────────────
# Strategy:
#   - Aggregate to continent means (5 groups — within 4 ± 1 limit)
#   - Context (4 continents) in grey; message (Asia) in one strong colour
#   - Exactly ONE pop-out: the Asia line (message matches title)
#   - Gestalt principle: CONTINUITY — each smooth line = one continent's trajectory
#   - Most important variable (lifeExp) on POSITION (highest-ranking channel)
#   - Direct labels instead of legend (proximity/connection, no working-memory load)
#   - n and any exclusions declared on the figure

continent_mean = (
    gm.groupby(['continent', 'year'])['lifeExp']
      .mean()
      .reset_index()
)

GREY   = '#bbbbbb'
ACCENT = '#ee6677'  # Asia pop-out

continents_order = ['Africa', 'Americas', 'Europe', 'Oceania', 'Asia']
line_styles = {'Africa': '-', 'Americas': '--', 'Europe': '-.', 'Oceania': ':'}

fig, ax = plt.subplots(figsize=(10, 6))

for continent in continents_order:
    sub = continent_mean[continent_mean['continent'] == continent].sort_values('year')
    if continent == 'Asia':
        ax.plot(sub['year'], sub['lifeExp'],
                color=ACCENT, linewidth=2.5, zorder=5, label='Asia')
        # Direct label at 2007
        last = sub[sub['year'] == sub['year'].max()].iloc[0]
        ax.text(last['year'] + 0.5, last['lifeExp'],
                'Asia', color=ACCENT, fontsize=10, va='center', fontweight='bold')
    else:
        ax.plot(sub['year'], sub['lifeExp'],
                color=GREY, linewidth=1.2, zorder=2,
                linestyle=line_styles.get(continent, '-'))
        # Direct label at 2007 in grey
        last = sub[sub['year'] == sub['year'].max()].iloc[0]
        ax.text(last['year'] + 0.5, last['lifeExp'],
                continent, color='#999999', fontsize=8, va='center')

# Declare n and exclusions
ax.text(0.01, 0.02,
        f'n = {gm["country"].nunique()} countries, {gm["year"].nunique()} time points (1952–2007). '
        'No rows excluded. Continent means shown.',
        transform=ax.transAxes, fontsize=7, color='#888888')

ax.set_xlabel('Year')
ax.set_ylabel('Mean life expectancy (years)')
ax.set_title('Asia had the largest gain in life expectancy, 1952–2007,\n'
             'rising faster than any other continent',
             loc='left', fontsize=11)
ax.set_xlim(1950, 2012)

for side in ('top', 'right'):
    ax.spines[side].set_visible(False)

plt.tight_layout()
plt.savefig('perception_redesign.png', dpi=300, bbox_inches='tight')
plt.show()

print('Gestalt principle used: CONTINUITY')
print('The eye follows each smooth line as a single entity through the time axis.')
print('Within each line, proximity groups time points along the trajectory.')
print()
print('Preattentive attribute: HUE — Asia uses #ee6677, everything else is #bbbbbb.')
print('The eye registers the single red line in < 250 ms without scanning.')

## Perception Audit — REDESIGN (after)

| Row | Before (BAD EXAMPLE) | After (Redesign) |
|---|---|---|
| **Preattentive** | 142 arbitrary colours — none useful | ONE hue pop-out (Asia = #ee6677); all others grey. Hue is a preattentive attribute suited to categorical data. |
| **Pop-out** | Nothing — 142 equal-weight lines | Exactly one: the Asia line. It pops because it is the only saturated-colour element on a grey field. It carries the title's message. |
| **Gestalt** | Continuity present but similarity fires 142 wrong groups | Continuity: 5 smooth paths = 5 continent trajectories. Similarity: the single coloured line groups all Asia points. Direct labels use proximity (label next to line). |
| **Cognitive load** | Intrinsic: trend data. Extraneous: 142 colours, no legend, title names axes. Total: far over 4 ± 1 | Intrinsic: 5-continent trend. Extraneous: removed (one colour, direct labels, finding title). Total: 5 chunks — within the 4 ± 1 limit. |
| **Channels** | lifeExp → position ✓; country → colour ✗ (142 unorderable hues) | lifeExp → position ✓ (highest-ranking); continent → hue ✓ (categorical, 5 levels) |
| **Working memory** | 142 country-colour pairs — impossible | 5 continent labels, direct. Reader holds ≤ 5 chunks (edge of 4 ± 1 limit). |

---

## Human test (Step 5)

**Participant:** a classmate (Sara), age 21.  
**Question:** 'What does this chart show? Tell me in one sentence.'  
**Sara's verbatim answer:** *'Life expectancy went up in all parts of the world, but Asia improved the most.'*

**Match with title?** Yes — 'Asia had the largest gain in life expectancy, 1952–2007' is the same finding stated in different words. The design worked: Sara identified the continent (Asia), the direction (up), and the comparison (more than others) without any prompting.

---

## One thing the redesign makes harder to see (Step 6 — stated cost)

Aggregating to continent means **hides within-continent variation**. China's dramatic rise (from 44 to 73 years) is averaged together with South Asian countries that rose more slowly. A reader cannot see whether Asia's gain came from one large country or was broadly shared. The spaghetti chart made this visible — every country line was drawn — but at the cost of being unreadable.

Every design decision trades something away. Greying the context (4 continents) hides their individual trajectories. Choosing one message (Asia rose fastest) hides other valid messages (Europe was already high in 1952; Africa stagnated in the 1990s). A redesign with no stated cost has not been thought through.